# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
38,0.66,759.5,318.5,220.5,3.5,0.0,9.750
755,0.69,735.0,294.0,220.5,3.5,0.4,15.475
593,0.79,637.0,343.0,147.0,7.0,0.4,41.255
581,0.90,563.5,318.5,122.5,7.0,0.4,35.980
179,0.69,735.0,294.0,220.5,3.5,0.1,12.770


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

>Teorik olarak, bitkileri öldürme riskini sınırlamak için modelimi MSE (Mean Squared Error) kayıp fonksiyonu üzerinde eğitirdim. Çünkü MSE, büyük tahmin hatalarını karesini alarak daha fazla cezalandırır. Büyük sıcaklık sapmaları bitkiler için daha ciddi sonuçlara yol açabileceğinden, modelin büyük hatalardan kaçınmasını sağlamak için MSE en uygun kayıp fonksiyonudur.

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [2]:
from sklearn.preprocessing import StandardScaler

X = data.drop(columns=["Average Temperature"])
y = data["Average Temperature"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [3]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate, KFold

model = SGDRegressor(loss="squared_error", random_state=42)

cv = KFold(n_splits=10, shuffle=True, random_state=42)

results = cross_validate(
    model,
    X_scaled,
    y,
    cv=cv,
    scoring="r2"
)

r2 = results["test_score"].mean()

r2

0.9082376983465839

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [4]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate, KFold

model = SGDRegressor(
    loss="squared_error",
    random_state=42
)

cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

results = cross_validate(
    model,
    X_scaled,
    y,
    cv=cv,
    scoring=["r2", "max_error"]
)

r2 = results["test_r2"].mean()
max_error_celsius = -results["test_max_error"].min()

r2, max_error_celsius

(0.9082376983465839, 9.168776887302624)

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [12]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate, KFold

model_mae = SGDRegressor(
    loss="epsilon_insensitive",
    epsilon=0,
    random_state=42
)

cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

results_mae = cross_validate(
    model_mae,
    X_scaled,
    y,
    cv=cv,
    scoring="r2"
)

r2_mae = results_mae["test_score"].mean()

r2_mae

0.8877292282436663

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [13]:
results_mae = cross_validate(
    model_mae,
    X_scaled,
    y,
    cv=cv,
    scoring="neg_max_error"
)

max_error_mae = -results_mae["test_score"].min()

max_error_mae

11.35551259498206

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

> Değerlendirdiğimiz modeller arasında görevimiz için MSE (En Küçük Kareler) kaybı ile eğitilen model daha uygundur. Çünkü sera bitkileri büyük sıcaklık sapmalarına karşı hassastır ve MSE büyük hataları karesini alarak daha fazla cezalandırır. Bu nedenle modelin büyük sıcaklık tahmin hatalarından kaçınmasını sağlamak, bitkilerin zarar görme riskini azaltmak açısından daha uygundur.

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [14]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/sena/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/sena/S16D4-S-loss-functions/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.63s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

